!pip install jax dask "dask[distributed]" --upgrade 

In [5]:
import os
os.chdir('..')
from src.bm25_fusion import BM25
# import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('stopwords')
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo

In [9]:
import random

categories = ["news", "science", "facts"]
sample_templates = [
    "The quick brown fox jumps over the lazy dog {}",
    "Never jump over the lazy dog quickly {}",
    "A quick brown dog outpaces a quick fox {}",
    "The lazy dog lies in the sun {}",
    "Quick brown foxes are faster than lazy dogs {}",
    "A lazy dog is a happy dog {}"
]

# Generate 1000 dummy documents.
documents = [
    random.choice(sample_templates).format(random.choice(categories))
    for _ in range(20000)
]

metadata_categories = [{"category": "news"}, {"category": "science"}, {"category": "facts"}]
metadata = [random.choice(metadata_categories) for _ in range(20000)]


In [11]:
bm25 = BM25(texts=documents,metadata=metadata,\
                variant="bm25+", stopwords={"is", "a", "the", "and"},chunk_size=1000,
    verbose=True,backend='dask' )

Tokenizing vocabulary: 100%|██████████| 20000/20000 [00:01<00:00, 19791.29it/s]


Tokenization complete. Memory usage: 293.33 MB


Computing TF matrix: 100%|██████████| 20000/20000 [00:00<00:00, 263831.70it/s]


TF Matrix Processed. Memory usage: 317.90 MB
IDF complete. Memory usage: 301.95 MB
Eager Indexing complete. Memory usage: 300.70 MB


In [12]:
results = bm25.query(["lazy","fox"],metadata_filter={"category": "news"}, top_k=2,do_keyword=False)
results

[{'text': 'A quick brown dog outpaces a quick fox science',
  'score': 9.431742668151855,
  'category': 'news'},
 {'text': 'A quick brown dog outpaces a quick fox news',
  'score': 9.431742668151855,
  'category': 'news'}]

In [13]:
for node in results:
    node1 = TextNode(text=node['text'], metadata=node)
    print(node1)
    print(node1.metadata)
    print("\n")

Node ID: 9c5c7273-d358-4aa0-947c-d1072d8dfc0b
Text: A quick brown dog outpaces a quick fox science
{'text': 'A quick brown dog outpaces a quick fox science', 'score': 9.431742668151855, 'category': 'news'}


Node ID: ddb66ee6-cccf-4488-b972-897d4f3d562b
Text: A quick brown dog outpaces a quick fox news
{'text': 'A quick brown dog outpaces a quick fox news', 'score': 9.431742668151855, 'category': 'news'}




In [14]:
bm25.save("bm25_model.pkl.gz")
bm25.save_hdf5("bm25_model.h5")

In [ ]:
bm25 = BM25.load("bm25_model.pkl.gz")
results = bm25.query(["fox"], metadata_filter={"category": "news"}, top_k=2)
results

[{'text': 'A quick brown dog outpaces a quick fox science',
  'score': 10.431742668151855,
  'category': 'news'},
 {'text': 'A quick brown dog outpaces a quick fox facts',
  'score': 10.431742668151855,
  'category': 'news'}]

In [18]:
bm25.verbose = True
bm25.add_document(["The quick brown egg jumps over the lazy dog news"], [{"category": "facts"}])

Computing TF matrix: 100%|██████████| 20002/20002 [00:00<00:00, 244732.99it/s]


In [19]:

results = bm25.query(["egg"], metadata_filter={"category": "facts"}, top_k=2)
results

[{'text': 'The quick brown egg jumps over the lazy dog news',
  'score': 10.177712440490723,
  'category': 'facts'},
 {'text': 'The quick brown egg jumps over the lazy dog news',
  'score': 10.177712440490723,
  'category': 'facts'}]

In [20]:
bm25_hdf5 = BM25.load_hdf5("bm25_model.h5")
bm25_hdf5.verbose = True
bm25_hdf5.add_document(["The quick brown egg jumps over the lazy fox news"], [{"category": "facts"}])
results = bm25_hdf5.query(["fox"], metadata_filter={"category": ["news","facts"]}, top_k=3)
results

Tokenizing vocabulary:   0%|          | 0/20001 [00:00<?, ?it/s]

Computing TF matrix: 100%|██████████| 20001/20001 [00:00<00:00, 284111.85it/s]


[{'text': 'Quick brown foxes are faster than lazy dogs news',
  'score': 10.723193168640137,
  'category': 'news'},
 {'text': 'Quick brown foxes are faster than lazy dogs facts',
  'score': 10.723193168640137,
  'category': 'facts'},
 {'text': 'A quick brown dog outpaces a quick fox news',
  'score': 10.723193168640137,
  'category': 'facts'}]